# DakiKobo — SCOLD / foundation embedding retrieval lab

Research notebook (Colab). **Not** production Flask code.

Goal: test whether image embeddings (e.g. SCOLD or another open vision encoder)
can retrieve similar leaf photos and few-shot classify better than chance —
**before** any training.

Candidate model: [enalis/scold](https://huggingface.co/enalis/scold)

Rules:
- Compare against Gemini screening from notebook 01 / production.
- Include blurry and not-a-plant negatives.
- Do not ship retrieval as diagnosis.
- API keys / tokens only via Colab secrets.


## 0. Setup

Optional installs in Colab. Prefer CPU first; GPU only if embedding a large set.


In [ ]:
# !pip -q install torch torchvision transformers pillow pandas scikit-learn
# !pip -q install huggingface_hub

import sys
from pathlib import Path

# If running from the repo root:
ROOT = Path.cwd()
if (ROOT / "scripts" / "vision_eval_helpers.py").exists():
    sys.path.insert(0, str(ROOT))
elif (ROOT.parent / "scripts" / "vision_eval_helpers.py").exists():
    sys.path.insert(0, str(ROOT.parent))

from scripts.vision_eval_helpers import (
    PhotoCase,
    load_manifest_csv,
    rank_by_embedding,
    retrieval_accuracy,
    summarize_predictions,
    write_markdown_report,
)

print("helpers ready")


## 1. Load eval manifest

Reuse the phone-photo / public-set manifest from notebook 01.


In [ ]:
MANIFEST = Path("reports/vision_eval_manifest.csv")  # adjust path
# cases = load_manifest_csv(MANIFEST)
# print(len(cases), "cases")
cases = []  # fill after you have a real manifest


## 2. Embed images (SCOLD or other)

This cell is a **stub**: swap in the real SCOLD / CLIP / DINOv2 encoder you choose.
Keep the interface `embed_image(path) -> list[float]` so the ranking helpers stay fixed.


In [ ]:
# Concrete Colab scaffold (optional GPU). Keep production on Gemini until this wins.
# Model id is a *candidate* — verify card/license on Hugging Face before heavy use.
# SCOLD: https://huggingface.co/enalis/scold
# Fallback for ablations: a public vision encoder (e.g. DINOv2 / CLIP) via the same interface.

from __future__ import annotations

MODEL_ID = "enalis/scold"  # change only after checking the model card
_device = "cpu"
_model = None
_processor = None


def load_encoder(model_id: str = MODEL_ID, device: str = "cpu"):
    """Lazy-load image encoder. Raises ImportError if deps missing."""
    global _model, _processor, _device
    if _model is not None and model_id == MODEL_ID:
        return _model, _processor
    import torch
    from transformers import AutoImageProcessor, AutoModel

    _device = device
    _processor = AutoImageProcessor.from_pretrained(model_id)
    _model = AutoModel.from_pretrained(model_id)
    _model.to(device)
    _model.eval()
    return _model, _processor


def embed_image(image_path: str) -> list[float]:
    """Return a dense embedding for one leaf photo.

    Interface required by vision_eval_helpers.rank_by_embedding / retrieval_accuracy.
    """
    from PIL import Image
    import torch

    model, processor = load_encoder()
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt")
    inputs = {k: v.to(_device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        # Prefer pooler_output when present; else mean-pool last_hidden_state.
        if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            vec = outputs.pooler_output[0]
        else:
            hidden = outputs.last_hidden_state
            vec = hidden.mean(dim=1)[0]
        vec = torch.nn.functional.normalize(vec.float(), dim=0)
    return vec.cpu().tolist()


print("embed_image defined — call load_encoder() once before batching.")
print("Ship rule: beats Gemini on phone photos + safe refusal on blurry/not-a-plant.")


## 3. Few-shot retrieval accuracy

Leave-one-out ranking with pure-Python helpers (unit-tested offline).


In [ ]:
# queries = [(c.case_id, embed_image(c.image_path), c.gold_label) for c in cases]
# metrics = retrieval_accuracy(queries, gallery, label_by_id, top_k=3)
# print(metrics["accuracy"], metrics["n"])
# write_markdown_report(
#     "reports/scold_retrieval_report.md",
#     "SCOLD retrieval lab",
#     {k: metrics[k] for k in ("n", "accuracy")},
#     notes="Compare to Gemini phone-photo accuracy from notebook 01.",
# )
print("Run after embed_image is implemented and cases are loaded.")


## 4. Compare to Gemini (from notebook 01)

Only promote a custom path if:
1. Retrieval/few-shot accuracy beats Gemini on **real phone photos**, and
2. Safe refusal on blurry / not-a-plant stays high, and
3. French field language remains hedged.

Otherwise keep Gemini Vision in production.


In [ ]:
print("Ship/no-ship: see write_markdown_report checklist in helpers.")


In [ ]:
# After MANIFEST is filled and images exist on disk:
# from scripts.vision_eval_helpers import retrieval_accuracy, write_markdown_report
#
# cases = load_manifest_csv(MANIFEST)
# gallery = []
# label_by_id = {}
# for case in cases:
#     try:
#         gallery.append((case.case_id, embed_image(case.image_path)))
#         label_by_id[case.case_id] = case.gold_label
#     except Exception as e:
#         print("skip", case.case_id, e)
# queries = [(cid, vec, label_by_id[cid]) for cid, vec in gallery]
# metrics = retrieval_accuracy(queries, gallery, label_by_id, top_k=3)
# print(metrics["accuracy"], metrics["n"])
# write_markdown_report(
#     "reports/scold_retrieval_report.md",
#     "SCOLD retrieval lab",
#     {"n": metrics["n"], "accuracy": metrics["accuracy"]},
#     notes="Compare to Gemini phone-photo metrics from notebook 01.",
# )
print("Uncomment the batch loop when you have a phone-photo manifest.")
